# 🔍 Comparador de Carpetas — Detector de Archivos Duplicados

**Objetivo:** Comparar dos carpetas del sistema, detectar archivos duplicados y opcionalmente eliminarlos de forma segura.

---

## 📌 Índice
1. [Importaciones y Configuración](#1-importaciones)
2. [Funciones Auxiliares](#2-auxiliares)
3. [Escaneo de Carpetas](#3-escaneo)
4. [Detección de Duplicados](#4-duplicados)
5. [Eliminación de Archivos](#5-eliminacion)
6. [Exportación de Resultados](#6-exportacion)
7. [Demo de uso sin interfaz gráfica](#7-demo)
8. [Cómo ejecutar la interfaz Streamlit](#8-streamlit)

---

## ⚙️ Instalación de dependencias
```bash
pip install streamlit pandas openpyxl
```

## 1. Importaciones y Configuración <a id='1-importaciones'></a>

Utilizamos exclusivamente librerías estándar de Python más `pandas` para manejo de datos.
No se requieren dependencias complejas adicionales.

In [ ]:
# ── Librerías estándar de Python ──
import os               # Manejo de rutas y sistema de archivos
import hashlib          # Cálculo de hashes MD5 / SHA256
import logging          # Registro de eventos y errores
import io               # Escritura en memoria (para exportar archivos)
from datetime import datetime          # Para nombrar archivos exportados
from collections import defaultdict    # Diccionario con valor por defecto
from pathlib import Path               # Manejo moderno de rutas

# ── Librerías externas ──
import pandas as pd    # Manejo de datos tabulares y exportación

# ── Configuración de Logging ──
# Guarda eventos en archivo .log para auditoría
LOG_FILE = "comparador_carpetas.log"
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)

print("✅ Importaciones completadas correctamente")

## 2. Funciones Auxiliares <a id='2-auxiliares'></a>

### 2.1 `formatear_bytes()`
Convierte un número de bytes a una representación legible para humanos (KB, MB, GB).

In [ ]:
def formatear_bytes(size_bytes: int) -> str:
    """
    Convierte bytes a formato legible.
    
    Ejemplos:
        500       → '500 B'
        1500      → '1.46 KB'
        1048576   → '1.00 MB'
    """
    if size_bytes < 1024:
        return f"{size_bytes} B"
    elif size_bytes < 1024 ** 2:
        return f"{size_bytes / 1024:.2f} KB"
    elif size_bytes < 1024 ** 3:
        return f"{size_bytes / 1024**2:.2f} MB"
    else:
        return f"{size_bytes / 1024**3:.2f} GB"

# Prueba rápida
for tamaño in [500, 1500, 1_048_576, 1_073_741_824]:
    print(f"{tamaño:>15,} bytes → {formatear_bytes(tamaño)}")

### 2.2 `calcular_hash()`

Calcula el hash criptográfico de un archivo **leyendo en bloques de 64KB**.

**¿Por qué en bloques?**
- Archivos de varios GB no caben en RAM si se leen completos.
- Leer por bloques es eficiente y no afecta el resultado del hash.

**Algoritmos soportados:**
- `md5`: Más rápido, suficiente para detección de duplicados.
- `sha256`: Más seguro, recomendado para archivos críticos.

In [ ]:
def calcular_hash(filepath: str, algoritmo: str = "md5", bloque: int = 65536) -> str | None:
    """
    Calcula el hash de un archivo leyéndolo en bloques.
    
    Args:
        filepath  : Ruta completa al archivo.
        algoritmo : 'md5' o 'sha256'.
        bloque    : Tamaño del bloque en bytes (default 64KB).
    
    Returns:
        Hash hexadecimal (str) o None si hay error.
    """
    try:
        # Seleccionar algoritmo
        h = hashlib.md5() if algoritmo == "md5" else hashlib.sha256()
        
        with open(filepath, "rb") as f:
            # Leer el archivo en bloques para no saturar la RAM
            while chunk := f.read(bloque):
                h.update(chunk)
        
        return h.hexdigest()
    
    except PermissionError:
        logger.warning(f"Sin permisos para leer: {filepath}")
        return None
    except FileNotFoundError:
        logger.warning(f"Archivo no encontrado: {filepath}")
        return None
    except OSError as e:
        logger.error(f"OSError al calcular hash de {filepath}: {e}")
        return None


# Demo: calcular hash de este mismo notebook
ruta_demo = __file__ if '__file__' in dir() else None
if ruta_demo and os.path.exists(ruta_demo):
    print(f"MD5   → {calcular_hash(ruta_demo, 'md5')}")
    print(f"SHA256 → {calcular_hash(ruta_demo, 'sha256')}")
else:
    # Crear archivo temporal de prueba
    with open("test_hash.txt", "w") as f:
        f.write("Hola mundo - archivo de prueba")
    print(f"MD5    → {calcular_hash('test_hash.txt', 'md5')}")
    print(f"SHA256 → {calcular_hash('test_hash.txt', 'sha256')}")
    os.remove("test_hash.txt")

## 3. Escaneo de Carpetas <a id='3-escaneo'></a>

La función `escanear_carpeta()` recorre **recursivamente** una carpeta usando `os.walk()`,
recolectando la ruta, nombre y tamaño de cada archivo.

**Manejo de errores:**
- `PermissionError`: archivos del sistema sin permisos de lectura.
- `FileNotFoundError`: archivo eliminado entre el listado y el acceso.
- `OSError`: errores generales del sistema operativo.

In [ ]:
def escanear_carpeta(carpeta: str) -> list[dict]:
    """
    Escanea recursivamente una carpeta y retorna info de cada archivo.
    
    Args:
        carpeta: Ruta raíz a escanear.
    
    Returns:
        Lista de dicts con claves: ruta, nombre, tamaño, carpeta_origen.
    """
    archivos = []
    
    # os.walk genera (directorio_raiz, subdirectorios, archivos)
    for root, _, files in os.walk(carpeta):
        for nombre in files:
            ruta_completa = os.path.join(root, nombre)
            try:
                tamaño = os.path.getsize(ruta_completa)
                archivos.append({
                    "ruta": ruta_completa,
                    "nombre": nombre,
                    "tamaño": tamaño,
                    "carpeta_origen": carpeta,
                })
            except (PermissionError, FileNotFoundError, OSError) as e:
                logger.warning(f"No se pudo acceder a {ruta_completa}: {e}")
    
    return archivos


# Demo con una carpeta real (ajusta la ruta)
carpeta_demo = os.path.expanduser("~")  # Carpeta home del usuario
if os.path.isdir(carpeta_demo):
    archivos_demo = escanear_carpeta(carpeta_demo)
    print(f"📂 Carpeta escaneada: {carpeta_demo}")
    print(f"📄 Total archivos encontrados: {len(archivos_demo)}")
    if archivos_demo:
        print("\nPrimeros 3 archivos:")
        for a in archivos_demo[:3]:
            print(f"  {a['nombre']:40s} | {formatear_bytes(a['tamaño'])}")

## 4. Detección de Duplicados <a id='4-duplicados'></a>

### Estrategia de comparación en 2 pasos:

| Paso | Criterio | Velocidad | Precisión |
|------|----------|-----------|----------|
| 1 | Nombre + Tamaño | ⚡ Muy rápido | Media |
| 2 | Hash (MD5/SHA256) | 🐢 Más lento | Alta |

**Optimización clave:** se usa un **diccionario indexado** (`defaultdict`) para buscar coincidencias en O(1) en lugar de comparar todos contra todos (O(n²)).

**Cache de hashes:** Los hashes calculados se guardan en un diccionario para no recalcularlos si el mismo archivo aparece más de una vez.

In [ ]:
def encontrar_duplicados(
    archivos1: list[dict],
    archivos2: list[dict],
    usar_hash: bool = False,
    algoritmo_hash: str = "md5",
    progress_callback=None,
) -> list[dict]:
    """
    Compara dos listas de archivos y devuelve los duplicados.
    
    Args:
        archivos1        : Lista de archivos de la carpeta 1.
        archivos2        : Lista de archivos de la carpeta 2.
        usar_hash        : Si True, confirma con hash criptográfico.
        algoritmo_hash   : 'md5' o 'sha256'.
        progress_callback: Función(actual, total) para actualizar progreso.
    
    Returns:
        Lista de dicts con información de cada par duplicado.
    """
    # ── Paso 1: Indexar carpeta 2 por (nombre, tamaño) ──
    # Estructura: {(nombre, tamaño): [lista de archivos]}
    indice2 = defaultdict(list)
    for archivo in archivos2:
        clave = (archivo["nombre"], archivo["tamaño"])
        indice2[clave].append(archivo)

    duplicados = []
    total = len(archivos1)
    hash_cache: dict[str, str] = {}  # Cache: {ruta → hash}

    # ── Paso 2: Comparar archivos de carpeta 1 contra el índice ──
    for i, arch1 in enumerate(archivos1):
        clave = (arch1["nombre"], arch1["tamaño"])

        if progress_callback:
            progress_callback(i + 1, total)

        if clave not in indice2:
            continue  # No hay candidatos, saltar

        candidatos = indice2[clave]

        if not usar_hash:
            # Coincidencia por nombre + tamaño
            for arch2 in candidatos:
                duplicados.append({
                    "nombre": arch1["nombre"],
                    "ruta_carpeta1": arch1["ruta"],
                    "ruta_carpeta2": arch2["ruta"],
                    "tamaño_bytes": arch1["tamaño"],
                    "tamaño_legible": formatear_bytes(arch1["tamaño"]),
                    "metodo": "Nombre + Tamaño",
                })
        else:
            # Calcular hash de arch1 (con cache)
            hash1 = hash_cache.get(arch1["ruta"])
            if hash1 is None:
                hash1 = calcular_hash(arch1["ruta"], algoritmo_hash)
                if hash1:
                    hash_cache[arch1["ruta"]] = hash1

            if hash1 is None:
                continue  # No se pudo calcular, saltar

            for arch2 in candidatos:
                # Calcular hash de arch2 (con cache)
                hash2 = hash_cache.get(arch2["ruta"])
                if hash2 is None:
                    hash2 = calcular_hash(arch2["ruta"], algoritmo_hash)
                    if hash2:
                        hash_cache[arch2["ruta"]] = hash2

                if hash2 and hash1 == hash2:
                    duplicados.append({
                        "nombre": arch1["nombre"],
                        "ruta_carpeta1": arch1["ruta"],
                        "ruta_carpeta2": arch2["ruta"],
                        "tamaño_bytes": arch1["tamaño"],
                        "tamaño_legible": formatear_bytes(arch1["tamaño"]),
                        "metodo": f"Hash {algoritmo_hash.upper()}",
                        "hash": hash1,
                    })

    return duplicados


print("✅ Función encontrar_duplicados() definida")

## 5. Eliminación de Archivos <a id='5-eliminacion'></a>

### Modo Dry Run (🛡️ RECOMENDADO)
Cuando `dry_run=True`, la función **simula** la eliminación registrando lo que haría, sin borrar nada.

**Errores manejados durante eliminación:**
- `PermissionError`: Archivo en uso por otro proceso.
- `FileNotFoundError`: Ya fue eliminado antes de llegar aquí.
- `OSError`: Error general del sistema operativo.

In [ ]:
def eliminar_archivos(rutas: list[str], dry_run: bool = True) -> dict:
    """
    Elimina archivos con manejo de errores robusto.
    
    Args:
        rutas   : Lista de rutas completas a eliminar.
        dry_run : True = simular | False = eliminar realmente.
    
    Returns:
        Dict con:
            'eliminados': lista de rutas procesadas/eliminadas
            'errores'   : lista de mensajes de error
    """
    resultado = {"eliminados": [], "errores": []}

    for ruta in rutas:
        if dry_run:
            # Modo seguro: solo registrar sin borrar
            resultado["eliminados"].append(ruta)
            logger.info(f"[DRY RUN] Simularía eliminar: {ruta}")
        else:
            try:
                os.remove(ruta)
                resultado["eliminados"].append(ruta)
                logger.info(f"Eliminado: {ruta}")
            except PermissionError:
                msg = f"Sin permisos para eliminar: {ruta}"
                resultado["errores"].append(msg)
                logger.error(msg)
            except FileNotFoundError:
                msg = f"Archivo ya no existe: {ruta}"
                resultado["errores"].append(msg)
                logger.warning(msg)
            except OSError as e:
                msg = f"Error al eliminar {ruta}: {e}"
                resultado["errores"].append(msg)
                logger.error(msg)

    return resultado


# Demo en modo dry run (completamente seguro)
rutas_demo = ["C:/archivo1.txt", "C:/archivo2.jpg", "C:/documento.pdf"]
resultado_demo = eliminar_archivos(rutas_demo, dry_run=True)
print(f"[DRY RUN] Archivos procesados: {len(resultado_demo['eliminados'])}")
print(f"[DRY RUN] Errores: {len(resultado_demo['errores'])}")

## 6. Exportación de Resultados <a id='6-exportacion'></a>

Dos funciones para exportar el DataFrame de resultados:
- `exportar_csv()`: genera bytes UTF-8 listos para descarga.
- `exportar_excel()`: genera un `.xlsx` con hoja "Duplicados".

In [ ]:
def exportar_csv(df: pd.DataFrame) -> bytes:
    """Serializa un DataFrame a CSV (bytes UTF-8)."""
    return df.to_csv(index=False).encode("utf-8")


def exportar_excel(df: pd.DataFrame) -> bytes:
    """Serializa un DataFrame a Excel .xlsx (bytes) en memoria."""
    buffer = io.BytesIO()
    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="Duplicados")
    return buffer.getvalue()


# Demo: crear DataFrame de ejemplo y exportar
df_ejemplo = pd.DataFrame([
    {"Nombre": "foto.jpg", "Ruta Carpeta 1": "C:/docs/foto.jpg",
     "Ruta Carpeta 2": "D:/backup/foto.jpg", "Tamaño": "2.50 MB"},
    {"Nombre": "informe.pdf", "Ruta Carpeta 1": "C:/docs/informe.pdf",
     "Ruta Carpeta 2": "D:/backup/informe.pdf", "Tamaño": "1.20 MB"},
])

csv_bytes = exportar_csv(df_ejemplo)
print(f"CSV generado: {len(csv_bytes)} bytes")
print("Contenido CSV:")
print(csv_bytes.decode("utf-8"))

try:
    excel_bytes = exportar_excel(df_ejemplo)
    print(f"\nExcel generado: {len(excel_bytes)} bytes")
except ImportError:
    print("\n⚠️ Instala openpyxl para exportar a Excel: pip install openpyxl")

## 7. Demo de uso sin interfaz gráfica <a id='7-demo'></a>

Aquí puedes probar toda la lógica **directamente en el notebook** sin necesitar Streamlit.

> ⚠️ **Ajusta** `CARPETA_1` y `CARPETA_2` con rutas reales de tu sistema.

In [ ]:
# ════════════════════════════════════════════
# CONFIGURACIÓN — Cambia estas rutas
# ════════════════════════════════════════════
CARPETA_1 = r"C:\Users\TuUsuario\Documentos"    # ← Cambia aquí
CARPETA_2 = r"C:\Users\TuUsuario\Descargas"     # ← Cambia aquí
USAR_HASH = False          # True = comparación por hash (más lento)
ALGORITMO = "md5"          # 'md5' o 'sha256'
DRY_RUN   = True           # True = nunca elimina archivos reales
# ════════════════════════════════════════════

import os

# Verificar carpetas
for label, carpeta in [("Carpeta 1", CARPETA_1), ("Carpeta 2", CARPETA_2)]:
    estado = "✅ Existe" if os.path.isdir(carpeta) else "❌ NO existe"
    print(f"{label}: {carpeta} → {estado}")

if os.path.isdir(CARPETA_1) and os.path.isdir(CARPETA_2):
    print("\n🔎 Escaneando carpetas...")
    archivos1 = escanear_carpeta(CARPETA_1)
    archivos2 = escanear_carpeta(CARPETA_2)
    print(f"  Carpeta 1: {len(archivos1)} archivos")
    print(f"  Carpeta 2: {len(archivos2)} archivos")

    print("\n🔄 Buscando duplicados...")
    duplicados = encontrar_duplicados(
        archivos1, archivos2,
        usar_hash=USAR_HASH,
        algoritmo_hash=ALGORITMO,
    )

    print(f"\n📋 Duplicados encontrados: {len(duplicados)}")
    if duplicados:
        espacio = sum(d['tamaño_bytes'] for d in duplicados)
        print(f"💾 Espacio recuperable: {formatear_bytes(espacio)}")

        df_resultado = pd.DataFrame(duplicados)
        print("\nPrimeros 10 resultados:")
        display(df_resultado[["nombre", "tamaño_legible", "metodo"]].head(10))

        # Guardar CSV
        nombre_csv = f"duplicados_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df_resultado.to_csv(nombre_csv, index=False)
        print(f"\n✅ Resultados guardados en: {nombre_csv}")
    else:
        print("🎉 No se encontraron duplicados.")
else:
    print("\n⚠️ Ajusta las rutas CARPETA_1 y CARPETA_2 para ejecutar este demo.")

## 8. Cómo ejecutar la interfaz Streamlit <a id='8-streamlit'></a>

### 📦 Instalación
```bash
pip install streamlit pandas openpyxl
```

### ▶️ Ejecución
```bash
streamlit run app.py
```

Esto abrirá automáticamente el navegador en `http://localhost:8501`

### 📖 Ejemplo de uso
1. Ingresa la ruta de **Carpeta 1** (ej: `C:\Users\Juan\Documentos`)
2. Ingresa la ruta de **Carpeta 2** (ej: `D:\Backup\Documentos`)
3. (Opcional) Activa **"Comparar por Hash"** en el panel lateral para mayor precisión
4. Asegúrate de que **"Modo Dry Run"** esté activado la primera vez
5. Haz clic en **"🚀 Iniciar Comparación"**
6. Revisa los resultados en la tabla
7. Exporta a CSV o Excel si lo necesitas
8. Si decides eliminar: elige la carpeta de la que quieres borrar duplicados

---

### 🗂️ Estructura de archivos generados
```
proyecto/
├── app.py                         ← Aplicación principal Streamlit
├── comparador_carpetas.ipynb      ← Este notebook
├── comparador_carpetas.log        ← Log de eventos (auto-generado)
└── duplicados_YYYYMMDD_HHMMSS.csv ← Exportación (al descargar)
```

### ⚠️ Consideraciones importantes
- Siempre revisa los resultados **antes** de eliminar archivos
- El **Modo Dry Run** es tu mejor aliado: úsalo siempre en la primera ejecución
- Los logs se guardan en `comparador_carpetas.log` para auditoría
- Carpetas con miles de archivos pueden tardar algunos segundos